**Data Collection**      
***Phenomenon of Culturomics on case of Michael Jackson Wikipedia page***

This notebook documents how the two datasets used in this project were collected:
pageviews and revision history for the Wikipedia article "Michael Jackson".     



**Step 1 Import libraries**

This stepload the Python libraries needed for the rest of the notebook.

In [1]:
import time
import gzip
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta, UTC



**Step 2 Collection method for pageviews (2009 archive + 2015-present API)**

This step  explain, in plain terms, how the pageviews data was obtained.

Two sources were combined:
- For June 2009, Wikipedia did not yet have a pageviews API. Data for this period
  comes from the archived hourly dump files (`pagecounts-raw`), streamed directly
  from `dumps.wikimedia.org` and parsed on the fly, without saving them to disk.
- For July 2015 onward, data comes from the official Wikimedia REST Pageviews API,
  which returns one number per day directly.

Both sources were collected with a polite request pace (a few seconds between
requests), a proper identifying User-Agent header, and automatic retries when a
request failed or timed out. This collection was run as a standalone script
(`pageviews_collector.py`) and its output was saved as `michael_jackson_pageviews.csv`.
This notebook loads that finished file and checks that it is complete and correct.


In [2]:
pageviews_path = Path("michael_jackson_pageviews.csv")
pageviews = pd.read_csv(pageviews_path, parse_dates=["date"])
pageviews = pageviews.sort_values("date").reset_index(drop=True)
pageviews.head()


,date,views
0,2009-06-20,23148
1,2009-06-21,14649
2,2009-06-22,13891
3,2009-06-23,13422
4,2009-06-24,12747


**Step 3 Check the pageviews data**

This step run basic checks on the loaded file.

The Goal is to make sure the collection process worked and the data can be trusted.

The code must be able to answer to question:      
Are there missing values, duplicate dates, or gaps in the daily
sequence?

The third step of the pipeline is  a short summary confirming the data is complete, or flagging problems.

In [3]:
print("Rows:", len(pageviews))
print("Missing values:\n", pageviews.isnull().sum())
print("Duplicate dates:", pageviews["date"].duplicated().sum())
print("Date range:", pageviews["date"].min(), "->", pageviews["date"].max())

modern = pageviews[pageviews["date"] >= "2015-07-01"]
full_range = pd.date_range(modern["date"].min(), modern["date"].max(), freq="D")
missing_days = full_range.difference(modern["date"])
print("Missing days in the 2015-present daily series:", len(missing_days))


Rows: 4110
Missing values:
 date     0
views    0
dtype: int64
Duplicate dates: 0
Date range: 2009-06-20 00:00:00 -> 2026-09-09 00:00:00
Missing days in the 2015-present daily series: 0


## Step 4 — Collection method for revision history

**What we do:** explain how the revision history (edit timestamps and article
size) was obtained.

**Goal:** make the collection process transparent and reproducible.

**Question:** where does the full edit history of the article come from?

**Result:** an understanding of the method, before we load the final file.

The MediaWiki Action API (`action=query`, `prop=revisions`) was used to request
every revision of the article, in batches of 500, from the very first edit to
the most recent one. Requests were sent through an authenticated session (using
a Wikipedia Bot Password, loaded from a local, non-shared `.env` file) and were
paced with a delay between requests to stay within Wikipedia's API etiquette.
Progress was saved after every batch, so an interrupted run could resume without
losing data. This collection was run as a standalone script
(`revisions_collector_auth.py`) and its output was saved as
`michael_jackson_revisions.csv`. This notebook loads that finished file.


In [ ]:
revisions_path = Path("michael_jackson_revisions.csv")
revisions = pd.read_csv(revisions_path, parse_dates=["timestamp"])
revisions = revisions.sort_values("timestamp").reset_index(drop=True)
revisions.head()


## Step 5 — Check the revisions data

**What we do:** run basic checks on the loaded file.

**Goal:** make sure the collection process worked and the data can be trusted.

**Question:** are there missing values, duplicate revisions, or an incomplete
date range?

**Result:** a short summary confirming the data is complete, or flagging problems.


In [ ]:
print("Rows:", len(revisions))
print("Missing values:\n", revisions.isnull().sum())
print("Duplicate revision IDs:", revisions["revid"].duplicated().sum())
print("Date range:", revisions["timestamp"].min(), "->", revisions["timestamp"].max())
print("Is sorted by time:", revisions["timestamp"].is_monotonic_increasing)


## Step 6 — Save a clean working copy

**What we do:** save both checked datasets under simple, stable file names for
use in the Analysis notebook.

**Goal:** keep data collection and data analysis as two separate, independent
steps.

**Question:** what exactly will the Analysis notebook load?

**Result:** two ready-to-use CSV files, unchanged in content, confirmed clean.


In [ ]:
pageviews.to_csv("pageviews_clean.csv", index=False)
revisions.to_csv("revisions_clean.csv", index=False)
print("Saved: pageviews_clean.csv, revisions_clean.csv")
